# MOF Quest model evaluation

Evaluate the fixed 22-reaction panel over repeated rounds. The implementation is in `mofinder.evaluation.quest`; model groups and settings are in `configs/quest_evaluation.json`. The classification prompt is shared with dataset preparation.

## Inputs and setup

Install `python -m pip install -e ".[evaluation,notebook]"` from the repository root. The question panel and prompt are included. Run the cells in order; keep `RUN_EVALUATION = False` for local validation or saved-run analysis. All paths below are repository-relative.

| Input | Location | For another run |
| --- | --- | --- |
| Question panel, JSON | `benchmarks/mof_quest/questions.json` | Select `questions_file` in `configs/quest_evaluation.json`; retain question IDs, reference labels, and the eight condition fields. |
| Classification prompt, TXT | `prompts/dataset_classification.txt` | Shared with dataset preparation. |
| Model groups, JSON | `configs/quest_evaluation.json` | Choose `GROUP` below and model IDs accessible to your account. |
| Saved predictions, optional | `results/evaluation/mof_quest/<run>/mof_manual_eval_*.csv` | Set `RUN_DIR` in the final section to the containing directory. |

For a small live test, set the selected group's `rounds` to `1` before enabling evaluation. The default configuration uses 20 rounds. New results are saved in a run directory under `results/evaluation/mof_quest/`.

Implementation: [question evaluation and round summaries](../src/mofinder/evaluation/quest.py). See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "mofinder").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "mofinder").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from mofinder.evaluation.quest import load_config, validate_config, run_evaluation, analyze_results

config = load_config(PROJECT_ROOT / "configs/quest_evaluation.json")
GROUP = "latest_fine_tuned"
RUN_EVALUATION = False
validate_config(config, GROUP)


## API key

Place your API key in `OPENAI_API_KEY`, or enter it at the hidden prompt below. Enter your own model IDs in the configuration when using fine-tuned models. Set `RUN_EVALUATION = True` to start the selected group.


In [ ]:
import getpass
import os

if RUN_EVALUATION and not os.environ.get("OPENAI_API_KEY"):
    api_key = getpass.getpass("Enter your OpenAI API key: ").strip()
    if not api_key:
        raise ValueError("An API key is required to run evaluation.")
    os.environ["OPENAI_API_KEY"] = api_key
    del api_key


In [ ]:
if RUN_EVALUATION:
    result = await run_evaluation(config, GROUP)
    print(result["output_dir"])
    display(result["metrics"])


## Analyze a saved run

Choose a completed run directory. Analysis reads the saved predictions and makes no API requests. Accuracy, precision, recall, and F1 use rows with valid reference and predicted labels. Round counts report unscored responses; standard deviations use `ddof=0`.


In [ ]:
RUN_DIR = None  # Set to a completed directory under results/evaluation/mof_quest/.

if RUN_DIR is not None:
    saved_csvs = sorted(Path(RUN_DIR).glob("mof_manual_eval_*.csv"))
    if not saved_csvs:
        raise FileNotFoundError("No model evaluation CSV files found in RUN_DIR.")
    analysis = analyze_results(saved_csvs)
    display(analysis)
